In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import category_encoders as ce

---

#### **01 ) - Importando os dados**

- 

---

In [16]:
df_d07_v1 = pd.read_csv("Input/D07_V1.csv")
df_d07_v1.head()


,1_age,2_workclass,3_final_weight,4_education,5_education_num,6_marital_status,7_occupation,8_relationship,9_race,10_sex,11_capital_gain,12_capital_loss,13_hours_per_week,14_native_country,15_income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [17]:
df_d07_status = pd.read_csv("Input/D07_status.csv")
df_d07_status.head()


,column,variable_type,variable_subtype,description,classification,transformation,status
0,1_age,quantitative,continuous,The age of the individual in years.,DEMOGRAPHIC,NaN,NaN
1,2_workclass,qualitative,nominal,The employment sector or status of the individ...,SOCIOECONOMIC,NaN,NaN
2,3_final_weight,quantitative,continuous,Final weight; a demographic weighting variable...,ERROR,NaN,NaN
3,4_education,qualitative,ordinal,The highest categorical level of education com...,SOCIOECONOMIC,NaN,NaN
4,5_education_num,quantitative,continuous,"The highest level of education completed, repr...",SOCIOECONOMIC,NaN,NaN


---

#### **02 ) - Split dos dados**

- 

---

In [18]:
y = df_d07_v1['15_income']
X = df_d07_v1.drop(columns=['15_income'])

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

---

#### **03 ) - Tratamento dos dados**

- 

---

#### 3.1 - *Quantitativas Contínuas*

In [20]:
# Lista de todas as variáveis quantitativas contínuas.
list_quant_con = df_d07_status[(df_d07_status['variable_type'] == 'quantitative') & (df_d07_status['variable_subtype'] == 'continuous')]['column'].to_list()
list_quant_con

['1_age',
 '3_final_weight',
 '5_education_num',
 '11_capital_gain',
 '12_capital_loss',
 '13_hours_per_week']

In [21]:
scaler = RobustScaler()
scaler.set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train[list_quant_con])
X_test_scaled = scaler.transform(X_test[list_quant_con])

#### 3.2 - *Qualitativas*

In [22]:
list_quali = df_d07_status[(df_d07_status['variable_type'] == 'qualitative')]['column'].to_list()
list_quali

['2_workclass',
 '4_education',
 '6_marital_status',
 '7_occupation',
 '8_relationship',
 '9_race',
 '10_sex',
 '14_native_country']

In [23]:
woe = ce.WOEEncoder(cols=list_quali)

X_train_woe = woe.fit_transform(X_train[list_quali], y_train)
X_test_woe = woe.transform(X_test[list_quali])

In [24]:
X_test_woe

,2_workclass,4_education,6_marital_status,7_occupation,8_relationship,9_race,10_sex,14_native_country
39567,-0.120645,-0.278436,-1.003629,-0.670707,-1.586666,0.077052,-0.947493,0.026444
26068,0.712967,-0.509652,-1.003629,-0.670707,-1.024874,0.077052,-0.947493,0.026444
3638,-0.120645,-0.509652,-1.878010,-0.059858,-1.024874,0.077052,0.326148,0.026444
25221,0.336558,-0.509652,-1.003629,-0.670707,-1.586666,0.077052,-0.947493,0.026444
6338,0.336558,-0.509652,-1.878010,0.409672,-2.970328,0.077052,0.326148,0.026444
...,...,...,...,...,...,...,...,...
32655,-0.120645,1.337382,-1.878010,-1.992853,-1.024874,-0.754882,0.326148,0.026444
44977,-0.120645,-0.278436,0.934681,-0.059858,0.942904,0.077052,0.326148,0.026444
22262,-0.120645,-0.509652,-1.878010,-0.882578,-2.970328,0.077052,0.326148,0.026444
28079,0.161762,-0.509652,-1.878010,-1.992853,-1.586666,-0.837406,-0.947493,0.026444


#### 3.2 - *Qualitativas Ordinais*

In [25]:
list_quali_ord = df_d07_status[(df_d07_status['variable_type'] == 'qualitative') & (df_d07_status['variable_subtype'] == 'ordinal')]['column'].to_list()
list_quali_ord

['4_education']

---

#### **04 ) - Consolidação dos dados tratados**

- 

---

In [26]:
# 1. Unindo as variáveis quantitativas contínuas e qualitativas nominais tratadas no teste
X_test_final = pd.concat([X_test_scaled, X_test_woe], axis=1)

# 2. Caso queira o conjunto de teste completo incluindo o alvo (15_income)
df_d07_test_final = pd.concat([X_test_final, y_test], axis=1)

In [27]:
# 1. Unindo as variáveis quantitativas contínuas e qualitativas nominais tratadas no treino
X_train_final = pd.concat([X_train_scaled, X_train_woe], axis=1)

# 2. Caso queira o conjunto de treino completo incluindo o alvo (15_income)
df_d07_train_final = pd.concat([X_train_final, y_train], axis=1)